In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_lfw_people
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)
X = lfw_people.data.astype("float32")
X = (X - X.min()) / (X.max() - X.min())
y = lfw_people.target
target_names = lfw_people.target_names

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    if i < len(lfw_people.images):
        ax.imshow(lfw_people.images[i], cmap="gray")
        ax.set_title(target_names[y[i]])
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
X_features = np.column_stack((X, X.min(axis=1), X.max(axis=1)))
y_categorical = to_categorical(y, num_classes=len(target_names))

X_train, X_test, y_train, y_test, y_train_labels, y_test_labels = train_test_split(
    X_features, y_categorical, y, test_size=0.3, random_state=42, stratify=y
)

print("Training data:", X_train.shape, y_train.shape)
print("Testing data:", X_test.shape, y_test.shape)

In [ ]:
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(7, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1
)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test loss: {test_loss:.3f}")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test_labels, y_pred, display_labels=target_names, xticks_rotation="vertical",
    cmap="Blues", ax=ax
)
ax.set_title("Original Model Confusion Matrix")
fig.tight_layout()
plt.show()

In [ ]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=0.3, random_state=42, stratify=y
)

pca = PCA(n_components=150, whiten=True, random_state=42)
X_train_pca = pca.fit_transform(X[train_idx])
X_test_pca = pca.transform(X[test_idx])
X_train_pca = np.column_stack((X_train_pca, X[train_idx].min(axis=1), X[train_idx].max(axis=1)))
X_test_pca = np.column_stack((X_test_pca, X[test_idx].min(axis=1), X[test_idx].max(axis=1)))
y_train_pca = y_categorical[train_idx]
y_test_pca = y_categorical[test_idx]
y_test_pca_labels = y[test_idx]

tuned_model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train_pca.shape[1],)),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(7, activation="softmax")
])
tuned_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
tuned_model.summary()

In [ ]:
tuned_history = tuned_model.fit(
    X_train_pca, y_train_pca, epochs=20, batch_size=32, validation_split=0.2, verbose=1
)
tuned_loss, tuned_accuracy = tuned_model.evaluate(X_test_pca, y_test_pca, verbose=0)
tuned_pred = np.argmax(tuned_model.predict(X_test_pca, verbose=0), axis=1)
print(f"Adjusted test accuracy: {tuned_accuracy:.3f}")
print(f"Adjusted test loss: {tuned_loss:.3f}")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test_pca_labels, tuned_pred, display_labels=target_names, xticks_rotation="vertical",
    cmap="Blues", ax=ax
)
ax.set_title("PCA-Adjusted Model Confusion Matrix")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for run, label in ((history, "Original"), (tuned_history, "PCA-adjusted")):
    axes[0].plot(run.history["accuracy"], label=f"{label} accuracy")
    axes[0].plot(run.history["val_accuracy"], linestyle="--", label=f"{label} val_accuracy")
    axes[1].plot(run.history["loss"], label=f"{label} loss")
    axes[1].plot(run.history["val_loss"], linestyle="--", label=f"{label} val_loss")
axes[0].set_title("Accuracy")
axes[1].set_title("Loss")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
plt.show()

1. Test accuracy is reported above for both models. The person with the most correct identifications is the class with the largest diagonal value in the confusion matrix.
2. Other useful features include HOG edges, facial landmarks, local binary patterns, and image augmentation such as small rotations or flips.